# 5. _Tokenização_ / Vetorização

Este notebook tem como objetivo realizar a vetorização/tokenização de enunciados e alternativas utilizando _embeddings_ de palavras. Ele inclui etapas de limpeza de dados, vetorização e armazenamento dos resultados para uso posterior em análises ou modelos de aprendizado de máquina.


In [1]:
# Importando Dependências para Vetorização
import pandas as pd
import numpy as np
import re
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
enem_df = pd.read_csv("../data/final/cleaned_enem_data.csv")
enem_df.head()

,numero_questao,enunciado,alternativas,gabarito,questao,pc_amostra_acertos,ano,gabarito_texto,distratores,enunciado_tokens,gabarito_tokens,distratores_tokens,dificuldade
0,1,A atmosfera terrestre é composta pelos gases n...,A: reduzir o calor irradiado pela Terra median...,C,1,0.93,2009,"reduzir o desmatamento, mantendo-se, assim, o ...",reduzir o calor irradiado pela Terra mediante ...,atmosfera terrestre composta gases nitrogênio ...,reduzir desmatamento mantendo assim potencial ...,reduzir calor irradiado terra mediante substit...,-1.70677
1,2,Analise a figura. Supondo que seja necessário ...,A: Concentração média de álcool no sangue ao l...,D,2,0.40,2009,Estimativa de tempo necessário para metaboliza...,Concentração média de álcool no sangue ao long...,analise figura supondo necessário dar título f...,estimativa tempo necessário metabolizar difere...,concentração média álcool sangue longo dia var...,0.62043
2,3,Estima-se que haja atualmente no mundo 40 milh...,"A: induzir a imunidade, para proteger o organi...",A,3,0.40,2009,"induzir a imunidade, para proteger o organismo...",ser capaz de alterar o genoma do organismo por...,estima atualmente mundo milhões pessoas infect...,induzir imunidade proteger organismo contamina...,capaz alterar genoma organismo portador induzi...,2.07704
3,4,"Em um experimento, preparou-se um conjunto de ...",A: os genótipos e os fenótipos idênticos.; B: ...,B,4,0.61,2009,os genótipos idênticos e os fenótipos diferentes.,os genótipos e os fenótipos idênticos.; difere...,experimento preparou conjunto plantas técnica ...,genótipos idênticos fenótipos diferentes,genótipos fenótipos idênticos diferenças genót...,0.11500
4,5,"Na linha de uma tradição antiga, o astrônomo g...",A: Ptolomeu apresentou as ideias mais valiosas...,E,5,0.60,2009,"Kepler apresentou uma teoria científica que, g...","Ptolomeu apresentou as ideias mais valiosas, p...",linha tradição antiga astrônomo grego ptolomeu...,kepler apresentou teoria científica graças mét...,ptolomeu apresentou ideias valiosas serem anti...,0.21694


In [31]:
def generate_word2vec_embeddings(data, model):
    not_found_words = set()
    result_embeddings = []

    # Precompilar regex para remover pontuações
    clean_ponctuation = re.compile(r"[.,:;()]")

    for item in data:
        word_vectors = []

        # Remove pontuações e divide em palavras
        words = clean_ponctuation.sub("", str(item)).split()

        for word in words:
            word_lower = word.lower()

            if word_lower in model:
                word_vectors.append(model[word_lower])
            else:
                not_found_words.add(word_lower)
        if word_vectors:
            # Calcula a média dos vetores
            mean_vector = np.mean(word_vectors, axis=0)
            result_embeddings.append(mean_vector)
        else:
            # Se não encontrou nenhuma palavra, adiciona um vetor nulo
            result_embeddings.append(np.zeros(model.vector_size))

    return result_embeddings, not_found_words

---

## 5.1. Word2Vec - 300 Dimensões


In [5]:
# Modelo de Embedding: Word2Vec NILC
# Download disponível em:
# http://nilc.icmc.usp.br/nilc/index.php/repositorio-de-word-embeddings-do-nilc
model_300 = KeyedVectors.load_word2vec_format("../embeddings/cbow_s300.txt")

### 5.1.1. Vetorização dos Enunciados


In [32]:
enem_df["enunciado_embbedings_word2vec_300"], not_found_words_enunciado = (
    generate_word2vec_embeddings(enem_df["enunciado_tokens"], model_300)
)

In [33]:
len(not_found_words_enunciado)

155

### 5.1.2. Vetorização dos Gabaritos


In [35]:
enem_df["gabarito_embbedings_word2vec_300"], not_found_words_gabarito = (
    generate_word2vec_embeddings(enem_df["gabarito_tokens"], model_300)
)

In [36]:
not_found_words_gabarito

{'anfifílica',
 'catalisem',
 'eflluente',
 'fotoimunoterapia',
 'fácia',
 'hexan',
 'leisnmaniose',
 'monofluoracético',
 'nitratação',
 'ondana',
 'penicilamina',
 'tetrassômicos',
 'trissômicos'}

### 5.1.3. Vetorização dos Distratores


In [37]:
enem_df["distratores_embbedings_word2vec_300"], not_found_words_distratores = (
    generate_word2vec_embeddings(enem_df["distratores_tokens"], model_300)
)

In [38]:
not_found_words_distratores

{'acondicionantes',
 'amonificação',
 'anfifílicos',
 'anfotérica',
 'apocinaceae',
 'borrifaria',
 'canade',
 'caramelizam',
 'cicloexanol',
 'codominante',
 'convergidos',
 'cromatofilia',
 'ddois',
 'eterificação',
 'fotoquimicamente',
 'fácia',
 'gãs',
 'hemozoínas',
 'hexan',
 'hexanal',
 'hexanoico',
 'interpopulacional',
 'linearizadas',
 'microvespa',
 'monofluoracético',
 'nitrosação',
 'owudomr',
 'penicilamina',
 'pirossulfato',
 'polialélico',
 'poligênico',
 'polipirrol',
 'polipoidia',
 'poliuretana',
 'precessionar',
 'resfriaria',
 'salinificação',
 'sinfilia',
 'solubilizado',
 'tetrassômicos',
 'trissômicos',
 'volatilizando',
 'vígula'}

### 5.1.4. Cálculo das Similaridades

In [39]:
# Calculando a similaridade de cosseno entre os embeddings de 300 dim
# Similaridade entre enunciado e gabarito
enem_df["similaridade_enunciado_gabarito_300"] = [
    cosine_similarity([enunciado], [gabarito])[0][0]
    for enunciado, gabarito in zip(
        enem_df["enunciado_embbedings_word2vec_300"],
        enem_df["gabarito_embbedings_word2vec_300"],
    )
]

# Similaridade entre enunciado e distratores
enem_df["similaridade_enunciado_distratores_300"] = [
    cosine_similarity([enunciado], [distratores])[0][0]
    for enunciado, distratores in zip(
        enem_df["enunciado_embbedings_word2vec_300"],
        enem_df["distratores_embbedings_word2vec_300"],
    )
]

# Similaridade entre gabarito e distratores
enem_df["similaridade_gabarito_distratores_300"] = [
    cosine_similarity([gabarito], [distratores])[0][0]
    for gabarito, distratores in zip(
        enem_df["gabarito_embbedings_word2vec_300"],
        enem_df["distratores_embbedings_word2vec_300"],
    )
]

enem_df[
    [
        "similaridade_enunciado_gabarito_300",
        "similaridade_enunciado_distratores_300",
        "similaridade_gabarito_distratores_300",
    ]
].head()

,similaridade_enunciado_gabarito_300,similaridade_enunciado_distratores_300,similaridade_gabarito_distratores_300
0,0.557157,0.780485,0.697827
1,0.303322,0.265094,0.621600
2,0.504171,0.595876,0.667647
3,0.374935,0.520597,0.820201
4,0.362750,0.657637,0.573542


---

## 5.2. Word2Vec - 100 Dimensões


In [13]:
# Modelo de Embedding: Word2Vec NILC
# Download disponível em:
# http://nilc.icmc.usp.br/nilc/index.php/repositorio-de-word-embeddings-do-nilc
model_100 = KeyedVectors.load_word2vec_format("../embeddings/cbow_s100.txt")

### 5.2.1. Vetorização dos Enunciados


In [40]:
enem_df["enunciado_embbedings_word2vec_100"], not_found_words_enunciado_100 = (
    generate_word2vec_embeddings(enem_df["enunciado_tokens"], model_100)
)

In [41]:
len(not_found_words_enunciado_100)

155

### 5.2.2. Vetorização dos Gabaritos


In [42]:
enem_df["gabarito_embbedings_word2vec_100"], not_found_words_gabarito_100 = (
    generate_word2vec_embeddings(enem_df["gabarito_tokens"], model_100)
)
not_found_words_gabarito_100

{'anfifílica',
 'catalisem',
 'eflluente',
 'fotoimunoterapia',
 'fácia',
 'hexan',
 'leisnmaniose',
 'monofluoracético',
 'nitratação',
 'ondana',
 'penicilamina',
 'tetrassômicos',
 'trissômicos'}

### 5.2.3. Vetorização dos Distratores


In [43]:
enem_df["distratores_embbedings_word2vec_100"], not_found_words_distratores_100 = (
    generate_word2vec_embeddings(enem_df["distratores_tokens"], model_100)
)
len(not_found_words_distratores_100)

43

### 5.2.4. Cálculo das Similaridades

In [44]:
# Calculando a similaridade de cosseno entre os embeddings de 100 dim
# Similaridade entre enunciado e gabarito
enem_df["similaridade_enunciado_gabarito_100"] = [
    cosine_similarity([enunciado], [gabarito])[0][0]
    for enunciado, gabarito in zip(
        enem_df["enunciado_embbedings_word2vec_100"],
        enem_df["gabarito_embbedings_word2vec_100"],
    )
]

# Similaridade entre enunciado e distratores
enem_df["similaridade_enunciado_distratores_100"] = [
    cosine_similarity([enunciado], [distratores])[0][0]
    for enunciado, distratores in zip(
        enem_df["enunciado_embbedings_word2vec_100"],
        enem_df["distratores_embbedings_word2vec_100"],
    )
]

# Similaridade entre gabarito e distratores
enem_df["similaridade_gabarito_distratores_100"] = [
    cosine_similarity([gabarito], [distratores])[0][0]
    for gabarito, distratores in zip(
        enem_df["gabarito_embbedings_word2vec_100"],
        enem_df["distratores_embbedings_word2vec_100"],
    )
]

enem_df[
    [
        "similaridade_enunciado_gabarito_100",
        "similaridade_enunciado_distratores_100",
        "similaridade_gabarito_distratores_100",
    ]
].head()

,similaridade_enunciado_gabarito_100,similaridade_enunciado_distratores_100,similaridade_gabarito_distratores_100
0,0.694265,0.860440,0.817126
1,0.356290,0.327731,0.745641
2,0.614781,0.708427,0.747481
3,0.451360,0.594306,0.854987
4,0.431371,0.765725,0.577122


---

## 5.3. Word2Vec - 50 Dimensões


In [22]:
# Modelo de Embedding: Word2Vec NILC
# Download disponível em:
# http://nilc.icmc.usp.br/nilc/index.php/repositorio-de-word-embeddings-do-nilc
model_50 = KeyedVectors.load_word2vec_format("../embeddings/cbow_s50.txt")

### 5.3.1. Vetorização dos Enunciados


In [45]:
enem_df["enunciado_embbedings_word2vec_50"], not_found_words_enunciado_50 = (
    generate_word2vec_embeddings(enem_df["enunciado_tokens"], model_50)
)
len(not_found_words_enunciado_50)

155

### 5.3.2. Vetorização dos Gabaritos


In [46]:
enem_df["gabarito_embbedings_word2vec_50"], not_found_words_gabarito_50 = (
    generate_word2vec_embeddings(enem_df["gabarito_tokens"], model_50)
)
not_found_words_gabarito_50

{'anfifílica',
 'catalisem',
 'eflluente',
 'fotoimunoterapia',
 'fácia',
 'hexan',
 'leisnmaniose',
 'monofluoracético',
 'nitratação',
 'ondana',
 'penicilamina',
 'tetrassômicos',
 'trissômicos'}

### 5.3.3. Vetorização dos Distratores


In [47]:
enem_df["distratores_embbedings_word2vec_50"], not_found_words_distratores_50 = (
    generate_word2vec_embeddings(enem_df["distratores_tokens"], model_50)
)
len(not_found_words_distratores_50)

43

### 5.3.4. Cálculo das Similaridades

In [48]:
# Calculando a similaridade de cosseno entre os embeddings de 50 dim
# Similaridade entre enunciado e gabarito
enem_df["similaridade_enunciado_gabarito_50"] = [
    cosine_similarity([enunciado], [gabarito])[0][0]
    for enunciado, gabarito in zip(
        enem_df["enunciado_embbedings_word2vec_50"],
        enem_df["gabarito_embbedings_word2vec_50"],
    )
]

# Similaridade entre enunciado e distratores
enem_df["similaridade_enunciado_distratores_50"] = [
    cosine_similarity([enunciado], [distratores])[0][0]
    for enunciado, distratores in zip(
        enem_df["enunciado_embbedings_word2vec_50"],
        enem_df["distratores_embbedings_word2vec_50"],
    )
]

# Similaridade entre gabarito e distratores
enem_df["similaridade_gabarito_distratores_50"] = [
    cosine_similarity([gabarito], [distratores])[0][0]
    for gabarito, distratores in zip(
        enem_df["gabarito_embbedings_word2vec_50"],
        enem_df["distratores_embbedings_word2vec_50"],
    )
]

enem_df[
    [
        "similaridade_enunciado_gabarito_50",
        "similaridade_enunciado_distratores_50",
        "similaridade_gabarito_distratores_50",
    ]
].head()

,similaridade_enunciado_gabarito_50,similaridade_enunciado_distratores_50,similaridade_gabarito_distratores_50
0,0.728683,0.867395,0.862642
1,0.428181,0.453869,0.808209
2,0.672166,0.781778,0.761080
3,0.554718,0.669521,0.872565
4,0.460525,0.812388,0.539009


---


---
## 5.4. Salvando Resultados


In [49]:
enem_df.to_pickle("../data/final/enem_data_embeddings.pkl")

In [50]:
enem_df.head()

,numero_questao,enunciado,alternativas,gabarito,questao,pc_amostra_acertos,ano,gabarito_texto,distratores,enunciado_tokens,...,distratores_embbedings_word2vec_100,similaridade_enunciado_gabarito_100,similaridade_enunciado_distratores_100,similaridade_gabarito_distratores_100,enunciado_embbedings_word2vec_50,gabarito_embbedings_word2vec_50,distratores_embbedings_word2vec_50,similaridade_enunciado_gabarito_50,similaridade_enunciado_distratores_50,similaridade_gabarito_distratores_50
0,1,A atmosfera terrestre é composta pelos gases n...,A: reduzir o calor irradiado pela Terra median...,C,1,0.93,2009,"reduzir o desmatamento, mantendo-se, assim, o ...",reduzir o calor irradiado pela Terra mediante ...,atmosfera terrestre composta gases nitrogênio ...,...,"[0.06891932, -0.056490835, 0.032688133, -0.018...",0.694265,0.860440,0.817126,"[0.08906648, 0.18746194, 0.09568582, -0.018923...","[0.177859, 0.14451149, -0.033945624, 0.12138, ...","[0.16039853, 0.15997498, 0.05087997, 0.0837478...",0.728683,0.867395,0.862642
1,2,Analise a figura. Supondo que seja necessário ...,A: Concentração média de álcool no sangue ao l...,D,2,0.40,2009,Estimativa de tempo necessário para metaboliza...,Concentração média de álcool no sangue ao long...,analise figura supondo necessário dar título f...,...,"[0.13126856, -0.0030684425, 0.13112548, 0.0888...",0.356290,0.327731,0.745641,"[0.0029705844, -0.0016135853, 0.007862832, 0.0...","[0.09892542, 0.057256144, 0.0030154246, 0.0359...","[0.07989774, 0.17586634, 0.0062733707, 0.03847...",0.428181,0.453869,0.808209
2,3,Estima-se que haja atualmente no mundo 40 milh...,"A: induzir a imunidade, para proteger o organi...",A,3,0.40,2009,"induzir a imunidade, para proteger o organismo...",ser capaz de alterar o genoma do organismo por...,estima atualmente mundo milhões pessoas infect...,...,"[0.074030176, 0.0145144975, 0.075817436, -0.10...",0.614781,0.708427,0.747481,"[0.037406296, 0.11757571, 0.023855807, 0.08883...","[0.10099166, 0.1493785, 0.084694505, 0.0487238...","[0.030623527, 0.19797918, 0.10676692, 0.043068...",0.672166,0.781778,0.761080
3,4,"Em um experimento, preparou-se um conjunto de ...",A: os genótipos e os fenótipos idênticos.; B: ...,B,4,0.61,2009,os genótipos idênticos e os fenótipos diferentes.,os genótipos e os fenótipos idênticos.; difere...,experimento preparou conjunto plantas técnica ...,...,"[-0.0139715355, 0.07850413, 0.0008902701, 0.06...",0.451360,0.594306,0.854987,"[-0.029311944, 0.10550189, 0.05036179, -0.0101...","[0.017343253, 0.35315275, 0.20922275, -0.09720...","[-0.010322735, 0.35458034, 0.16574, 0.0417216,...",0.554718,0.669521,0.872565
4,5,"Na linha de uma tradição antiga, o astrônomo g...",A: Ptolomeu apresentou as ideias mais valiosas...,E,5,0.60,2009,"Kepler apresentou uma teoria científica que, g...","Ptolomeu apresentou as ideias mais valiosas, p...",linha tradição antiga astrônomo grego ptolomeu...,...,"[-0.026497144, -0.0024524825, 0.008412318, -0....",0.431371,0.765725,0.577122,"[0.04814419, 0.08036936, 0.0690699, -0.0082249...","[-0.1161132, 0.0615291, -0.0451972, -0.0884399...","[0.0076814005, 0.03361294, -0.0035526846, -0.0...",0.460525,0.812388,0.539009


---